In [2]:
import numpy as np
import scipy
from scipy.special import factorial
import math
from scipy.integrate import nquad

In [3]:
c=2.998*10**8 #in m/s

#in natural units:
v0 = 220*1000/c
v_esc = 500*1000/c 
v_E = 240*1000/c 

#in eV
w_minus = 55*10**-3
w_plus = 20*10**-3
m_Si_nucleus = 2.616e10
proton_mass= 938.272e6

dm_density = 0.3e9 #eV/cm^3, plus minus 0.1
target_density = 1.307e33 #eV/cm^3 from silicon density of 2329.6 kg/m3

#in 1/eV
a_Si = 2*np.pi/(2.28*1000) #Silicon lattice constant
a0 = 2.68e-4 #Bohr radius

ratio=v_esc/v0
N0 = v0**3*np.pi*(np.sqrt(np.pi)*scipy.special.erf(ratio)-2*ratio*np.exp(-ratio**2))
del(ratio)

In [4]:
class TwoSiteScattering():
    def __init__(self, A=28, V=(a_Si**3)/4, w1=w_minus, w2=w_plus, m_targ=m_Si_nucleus, lattice_spacing=a_Si):
        self.A = A
        self.V = V
        self.w1 = w1
        self.w2 = w2
        self.m_targ = m_targ
        self.a=lattice_spacing

    """
    The Debye-Waller function for a two-site lattice
    @params 
        q: the momentum transfer from dark matter to the crystal
    """
    def debye_waller(self, q):
        return q**2/(8*self.m_targ)*(1/self.w1+1/self.w2)
    
    
    """
    The diagonal terms (without the delta function) for one n,j combination
    @params
        q: the momentum transfer from dark matter to the crystal
        w1, w2: energy levels of the element
        n: principle phonon number
        j: phonon mode number
    """
    def diagonal_C_true(self, q, n, j):
        expDebyeWaller = np.exp(-2*self.debye_waller(q))
        c=expDebyeWaller/self.V * (q**2/(4*self.m_targ))**n * 2*np.pi/(factorial(j)*factorial(n-j)*self.w1**j*self.w2**(n-j))
        return c
    
    """
    The off-diagonal terms (without the delta function) for one n,j combination
    @params
        q: the momentum transfer from dark matter to the crystal
        n: principle phonon number
        j: phonon mode number
    """
    def offDiagonal_C_true(self, q, n, j):
        expDebyeWaller = np.exp(-2*self.debye_waller(q))
        c=expDebyeWaller/self.V * (q**2/(4*self.m_targ))**n * 2*np.pi/(factorial(j)*factorial(n-j)*self.w1**j*self.w2**(n-j))
        if (n-j)%2==1:
            c = -c
        c = c*np.cos(q*self.a)
        return c

    """
    Structure Factor for one n and j pair, integrated with omega to remove the delta function
    @params
        q: the momentum transfer from dark matter to the crystal
        n: principle phonon number
        j: phonon mode number
    """
    def StructureFactor_n_j(self,q,n, j):
        return 2*self.A**2*(self.diagonal_C_true(q, n, j)
                          +self.offDiagonal_C_true(q, n, j))
    
    """
    Structure Factor for one n, integrated with omega to remove the delta function
    @params
        q: the momentum transfer from dark matter to the crystal
        n: principle phonon number
    """
    def StructureFactor_n(self,q, n):
        return sum([2*self.A**2*(self.diagonal_C_true(q, n, j)
                            +self.offDiagonal_C_true(q, n, j)) for j in range(n+1)] )
    
    """
    Diagonal Structure Factor for one n, integrated with omega to remove the delta function
    @params
        q: the momentum transfer from dark matter to the crystal
        n: principle phonon number
    """
    def StructureFactor_diag(self, q, n):
        return sum([2*self.A**2* self.diagonal_C_true(q, n, j) for j in range(n+1)] )
    
    """
    This gives the two-body reduced mass equation
    @params:
        m1, m2: masses of the two bodies
    @returns: Effective mass of the two bodies
    """
    def reducedMass(self,m1, m2):
        return m1*m2/(m1+m2)
    
    """
    Helper function 1 and 2 for finding the output of the inverse mean function.
    @Params:
        v_min: input minimum speed for dark matter
        m_chi: mass of dark matter
        const: scaling value for graphing.  As a default, it gives the true scale of the equation
    """
    def inverseMeanFunction_sol1(self, v_min, m_chi, const):
        summ = np.sqrt(np.pi)*v0*(scipy.special.erf((v_min+v_E)/v0)-scipy.special.erf((v_min-v_E)/v0))
        summ -= 4*np.exp(-v_esc**2/v0**2)*v_E
        return summ*const
    def inverseMeanFunction_sol2(self, v_min, m_chi, const):
        summ = np.sqrt(np.pi)*v0*(scipy.special.erf(v_esc/v0)-scipy.special.erf((v_min-v_E)/v0))
        summ -= 2*np.exp(-v_esc**2/v0**2)*(v_esc-v_min+v_E)
        return summ*const

    """
    Returns the inverse mean value for any given v_min.  
    @Params:
        v_min: input minimum speed for dark matter
        m_chi: mass of dark matter
        const: scaling value for graphing.  As a default, it gives the true scale of the equation
    """
    def inverseMeanFunction(self, v_min, m_chi, const = v0**2*np.pi/(2*v_E*N0)):
        if v_min>v_esc+v_E:
            return 0
        elif v_min>v_esc-v_E:
            return self.inverseMeanFunction_sol2(v_min,m_chi, const)
        else: 
            return self.inverseMeanFunction_sol1(v_min,m_chi, const)
    
    

In [18]:
import matplotlib.pyplot as plt

In [15]:
runner = TwoSiteScattering()